# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the FAIR² dataset using the `mlcroissant` library. The notebook follows the Croissant standard, referencing all dataset entities by their unique `@id` fields.

### Dataset Source
The dataset is provided via a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the Croissant dataset metadata and instantiate the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and print summary
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print(f"Name: {meta.name}\n\nDescription: {meta.description}\n")

## 2. Data Overview
Explore the dataset structure - list all record sets, with their titles, field `@id`s, and column mappings.

> **All entity references are by `@id` fields as per the Croissant standard.**

In [ ]:
# List all available Record Sets
print("Available record sets (by @id):")
for rset in meta.record_sets:
    print(f"- {rset.id}: {rset.name if hasattr(rset, 'name') else ''}")

# Get details of each Record Set: fields and columns @id
for rset in meta.record_sets:
    print(f"\nRecordSet @id: {rset.id}")
    for field in rset.fields:
        print(f"  Field @id: {field.id}, Name: {getattr(field, 'name', '')}, Data type: {getattr(field, 'data_type', '')}")
        if hasattr(field, 'column') and field.column:
            print(f"    Column mapping(s) @id(s): {[col.id for col in (field.column if isinstance(field.column, list) else [field.column])]}")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis.

_Below extraction references record set and field IDs printed above. Modify as needed depending on the dataset structure._

In [ ]:
# Prepare a list of record set @id's to load
record_sets = [rset.id for rset in meta.record_sets]
# Optionally, focus on just the first for demonstration
# record_sets = [meta.record_sets[0].id]
dataframes = {}

for rset_id in record_sets:
    records = list(dataset.records(record_set=rset_id))
    if records:
        dataframes[rset_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from {rset_id}")
        print(f"Fields: {dataframes[rset_id].columns.tolist()}")
        print(dataframes[rset_id].head(2))
    else:
        print(f"No records found in record set {rset_id}")

# Select a record set to analyze further:
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nProceeding with main record set: {main_record_set_id}")
    main_df = dataframes[main_record_set_id]
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Now, perform some common data wrangling tasks using the record set and field `@id` as identifiers. Examples include filtering, normalization, and grouping.

_Edit `numeric_field_id` and `group_field_id` below to correspond to actual `@id`s as printed above. By default, these will attempt to auto-select numeric columns if available._

In [ ]:
import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Try to infer a numeric field by dtype
    numeric_columns = df.select_dtypes(include=[np.number]).columns
    if len(numeric_columns) == 0:
        # Fallback: try to auto-convert possible numeric fields
        df_num = df.apply(pd.to_numeric, errors='ignore')
        numeric_columns = df_num.select_dtypes(include=[np.number]).columns
        df = df_num
    if len(numeric_columns) > 0:
        numeric_field = numeric_columns[0]
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() else 1)
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a likely categorical field (not the numeric)
        group_candidates = [c for c in df.columns if c != numeric_field]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable field found to group by.")
    else:
        print("No numeric fields found for filtering and normalization.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize a numeric field distribution or its relationship with a group/categorical field.

_The below attempts automatic column selection; edit `numeric_field` and `group_field` if desired._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and 'numeric_field' in locals():
    fig, ax = plt.subplots(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, ax=ax)
    ax.set_title(f"Distribution of {numeric_field}")
    plt.show()

    if 'group_field' in locals() and group_field is not None:
        # Plot group means if there is a group field
        group_means = df.groupby(group_field)[numeric_field].mean(numeric_only=True)
        fig, ax = plt.subplots(figsize=(10,5))
        group_means.plot(kind='bar', ax=ax)
        ax.set_ylabel(f"Mean {numeric_field}")
        ax.set_title(f"Mean {numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore a Croissant-formatted FAIR² dataset using `mlcroissant`. You can use the provided record set and field `@id`s for programmatic exploration and analysis. 

Typical next steps: perform domain-specific analysis using grouped or normalized results and investigate individual proteins or peptides according to your research needs.

For more about the Croissant standard, see [https://mlcommons.org/croissant](https://mlcommons.org/croissant).